# ADT treatment intent -- localized/adjuvant vs. metastatic

Splits the ADT-entry cohort into patients treated with **adjuvant intent** for
localized disease and those treated for **metastatic** disease, using
medication history alone.

The rest of the pipeline anchors every patient at `TREATMENT_ANCHOR_DATE` --
the first ADT date -- and then treats the cohort as homogeneous. It is not.
Adjuvant ADT is 6-24 months alongside definitive local therapy and most such
patients are cured; metastatic ADT is indefinite castration ending in
resistance and death. Mixing them biases every survival model, because the
adjuvant group contributes long event-free follow-up unrelated to the
mCRPC/NEPC biology being modelled.

This notebook is **diagnostic and additive**. It writes a per-patient label to
its own directory and changes no cohort, no existing output, and no upstream
stage. Nothing downstream reads it unless you wire it in deliberately.

**Population.** This notebook reads `icd_prostate_mrn_flags.csv` -- one row per
ICD-C61 patient -- and classifies everyone flagged `ADT_EXPOSED`. That is
deliberately **wider than the modelled cohort**: `ADT_EXPOSED` means dated ADT
on/after prostate diagnosis and nothing else, so it keeps patients the survival
cohort excludes for PARPi exposure, pre-diagnosis platinum, competing cancers,
or fewer than five PSA tests. Adjuvant-intent patients are exactly the kind
likely to be dropped by those filters, so classifying only the eligible cohort
would bias the split it is meant to expose. `ELIGIBLE` is carried through so
any view can be narrowed back down.

**Prerequisite:** Stage 1 of `01_preprocessing.ipynb` (`cp.compile_cohort`),
which writes the flags file and `prostate_icd_data.csv`. Follow-up dates come
straight from the patient-status table, which covers the whole C61 population
rather than just the cohort.

**Stage 2 (`cp.preprocess_labs`, adt arm) is optional but unlocks the last
section**: testosterone and PSA trajectories from one year before ADT to five
years after, and Kaplan-Meier curves for the platinum, NEPC and AVPC
endpoints. Without it, everything through the death KM still runs.

## Configuration

Paths follow `compass_pipeline`, so this notebook picks up the same
`profile_data` sources and output root as the rest of COMPASS.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, ".")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl

from COMPASS.data_preprocessing.classify_adt_intent import (
    GAP_THRESHOLD_DAYS,
    MIN_FOLLOWUP_DAYS,
    classify_adt_intent,
    summarize_intent,
)
from COMPASS.data_preprocessing.compile_COMPASS_cohort_data import (
    load_patient_status,
)
from survival_common.plotting import overlay_km
from COMPASS.data_preprocessing.adt_intent_trajectories import (
    CASTRATE_NG_DL,
    CASTRATE_STRICT_NG_DL,
    INTENT_COLORS,
    KM_ENDPOINTS,
    PSA_LAB_NAME,
    PSA_LOG_FLOOR,
    TESTOSTERONE_LAB_NAME,
    build_death_km_input,
    build_km_input,
    build_lab_trajectory,
    km_series_by_intent,
    load_longitudinal,
    logrank_by_intent,
    plot_lab_trajectory,
    summarize_km,
    summarize_trajectory_coverage,
)
from COMPASS.data_preprocessing.validate_adt_intent import (
    compute_first_metastasis_icd_date,
    compute_psa_nadir_features,
    report_against_metastasis_icd,
    report_by_adt_start_year,
    report_contradictions,
    report_gap_sensitivity,
    report_survival,
    _rate_report,
)

DATA_ROOT = cp._PROFILE_OUTPUT_ROOT
MEDICATIONS_PATH = cp.PROFILE_SOURCES["MEDICATIONS"]
PATIENT_STATUS_PATH = cp.PROFILE_SOURCES["PT_INFO_STATUS_REGISTRATION"]

# Written by Stage 1 of 01_preprocessing.ipynb.
FLAGS_PATH = DATA_ROOT / "mrn_lists" / "icd_prostate_mrn_flags.csv"
ICD_PATH = DATA_ROOT / "prostate_icd_data.csv"

OUT_DIR = DATA_ROOT / "adt_intent"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"medications : {MEDICATIONS_PATH}")
print(f"pt status   : {PATIENT_STATUS_PATH}")
print(f"icd flags   : {FLAGS_PATH}  exists={FLAGS_PATH.exists()}")
print(f"icd record  : {ICD_PATH}  exists={ICD_PATH.exists()}")
print(f"output dir  : {OUT_DIR}")
print(f"\ndefaults: gap={GAP_THRESHOLD_DAYS}d  min_followup={MIN_FOLLOWUP_DAYS}d")

## Load the ICD-C61 flags and select the ADT-exposed population

`icd_prostate_mrn_flags.csv` is the audit table Stage 1 writes for every
ICD-C61 patient, with one column per cohort criterion. Its flags are computed
from the same sets that drive actual cohort membership, so they cannot drift
from the pipeline's own eligibility calculation.

`ADT_EXPOSED` is the flag to classify on: it marks dated ADT on/after prostate
diagnosis, which is precisely the population "ADT treatment intent" is a
question about.

In [ ]:
flags = pl.read_csv(FLAGS_PATH, infer_schema_length=0).with_columns(
    pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False),
    *[
        pl.col(c).cast(pl.Float64, strict=False).cast(pl.Int8, strict=False)
        for c in (
            "ADT_EXPOSED", "ELIGIBLE", "PARPI_EXPOSED", "HAS_5_OR_MORE_PSA_TESTS",
            "PLATINUM_BEFORE_DIAGNOSIS", "HAS_POST_ADT_EXCLUSION_CANCER",
            "HAS_NON_PROSTATE_PRIMARY", "ARPI_DOCETAXEL_EXPOSED",
            "DATED_PROSTATE_DIAGNOSIS", "MALE",
        )
    ],
).drop_nulls("DFCI_MRN")

print(f"ICD-C61 patients      : {flags.height:,}")
print(f"  ADT_EXPOSED         : {flags['ADT_EXPOSED'].sum():,}")
print(f"  ELIGIBLE (modelled) : {flags['ELIGIBLE'].sum():,}")

adt_exposed = flags.filter(pl.col("ADT_EXPOSED") == 1)

# How much wider than the modelled cohort this is, and why.
extra = adt_exposed.filter(pl.col("ELIGIBLE") == 0)
print(f"\nADT-exposed but not eligible: {extra.height:,}")
for col in (
    "HAS_5_OR_MORE_PSA_TESTS", "PARPI_EXPOSED", "PLATINUM_BEFORE_DIAGNOSIS",
    "HAS_POST_ADT_EXCLUSION_CANCER", "MALE", "DATED_PROSTATE_DIAGNOSIS",
):
    n = extra.filter(pl.col(col) == (0 if col.startswith(("HAS_5", "MALE", "DATED")) else 1)).height
    print(f"  {col:<32} {n:,}")

### Follow-up dates

Follow-up comes from the patient-status table rather than the survival cohort,
because that cohort only covers eligible patients and this notebook classifies
a wider population. `FOLLOW_UP_END_DATE` and `DEATH` are derived exactly as
`build_survival_cohort` derives them: death date when present, else last
contact.

Follow-up is not optional cosmetics. Without it a patient whose ADT started
shortly before the data cutoff is indistinguishable from one who completed a
short adjuvant course, and the adjuvant class silently fills with
recently-treated metastatic patients. The classifier forces every such patient
to `INDETERMINATE`.

In [ ]:
status = load_patient_status(PATIENT_STATUS_PATH)

follow_up = status.select(
    pl.col("DFCI_MRN"),
    pl.col("DEATH_DATE").fill_null(pl.col("LAST_CONTACT_DATE")).alias("FOLLOW_UP_END_DATE"),
    pl.col("DEATH_DATE").is_not_null().cast(pl.Int64).alias("DEATH"),
).filter(pl.col("DFCI_MRN").is_in(adt_exposed["DFCI_MRN"]))

print(f"ADT-exposed patients      : {adt_exposed.height:,}")
print(f"  with status row         : {follow_up.height:,}")
print(f"  with follow-up date     : {follow_up['FOLLOW_UP_END_DATE'].is_not_null().sum():,}")

missing = adt_exposed.height - follow_up['FOLLOW_UP_END_DATE'].is_not_null().sum()
if missing:
    print(f"\n{missing:,} ADT-exposed patients lack a follow-up date and will "
          f"fall to INDETERMINATE (censoring cannot be ruled out).")

meds = cp.scan_source(MEDICATIONS_PATH).collect().with_columns(
    pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
).filter(pl.col("DFCI_MRN").is_in(adt_exposed["DFCI_MRN"]))
print(f"\nmedication rows for ADT-exposed patients: {meds.height:,}")

## Classify

Rules fire in order, first match wins. `ADT_INTENT_REASON` records which one
fired, so every label can be traced back to its evidence.

| # | Rule | Label |
| --- | --- | --- |
| 1 | taxane / radium-223 / sipuleucel-T / estramustine / mitoxantrone exposure | `METASTATIC` |
| 2 | ARPI **and** ADT span > 1 year | `METASTATIC` |
| 3 | ADT span > 3 years over <= 2 episodes | `METASTATIC` |
| 4 | ADT ongoing at last contact **and** span > 2 years | `METASTATIC` |
| 5 | insufficient follow-up after ADT start | `INDETERMINATE` |
| 6 | single course <= 36 months, no escalation, not ongoing | `LOCALIZED_ADJUVANT` |
| 7 | anything else | `INDETERMINATE` |

An ARPI never classifies on its own (rule 2 requires sustained ADT):
apalutamide and darolutamide are approved in *non-metastatic* CRPC, and ARPIs
are increasingly used in high-risk localized disease.

In [ ]:
labelled = classify_adt_intent(meds, follow_up=follow_up)

# The classifier never sees DEATH -- it is held out for validation below.
labelled = labelled.join(
    follow_up.select("DFCI_MRN", "DEATH"), on="DFCI_MRN", how="left"
)

# Carry ELIGIBLE through so any downstream view can be narrowed back to the
# modelled cohort without re-reading the flags file.
labelled = labelled.join(
    adt_exposed.select("DFCI_MRN", "ELIGIBLE"), on="DFCI_MRN", how="left"
)

print(summarize_intent(labelled))
print()
print(
    labelled.group_by(["ADT_INTENT", "ADT_INTENT_REASON"])
    .agg(pl.len().alias("n_patients"))
    .sort("n_patients", descending=True)
)

### Duration profile by class

A sanity check on the core signal. `METASTATIC` should show a much longer
median ADT span than `LOCALIZED_ADJUVANT`; if the two are close, the duration
signal is not separating and the rest of the notebook is not worth reading.

In [ ]:
print(
    labelled.group_by("ADT_INTENT")
    .agg(
        pl.len().alias("n_patients"),
        pl.col("ADT_SPAN_DAYS").median().round(0).alias("median_span_days"),
        pl.col("ADT_N_EPISODES").median().alias("median_episodes"),
        pl.col("ADT_N_RECORDS").median().alias("median_records"),
        pl.col("FOLLOWUP_DAYS_FROM_ADT").median().round(0).alias("median_followup_days"),
        (pl.col("HAS_ARPI").cast(pl.Float64).mean() * 100).round(1).alias("pct_arpi"),
    )
    .sort("median_span_days")
)

### Eligible cohort vs. the full ADT-exposed population

The reason for classifying the wider population. If the intent mix differs
between `ELIGIBLE == 1` and the patients the cohort filters drop, then the
modelled cohort is not a random subset of ADT-treated prostate patients, and
the exclusions are themselves selecting on treatment intent.

A markedly higher `LOCALIZED_ADJUVANT` share among the dropped patients is the
expected direction -- adjuvant patients get fewer PSA tests and are more often
lost to the 5-PSA requirement.

In [ ]:
by_elig = (
    labelled.with_columns(
        pl.when(pl.col("ELIGIBLE") == 1)
        .then(pl.lit("eligible"))
        .otherwise(pl.lit("excluded"))
        .alias("cohort_status")
    )
    .group_by(["cohort_status", "ADT_INTENT"])
    .agg(pl.len().alias("n_patients"))
)

wide = by_elig.pivot(values="n_patients", index="ADT_INTENT", on="cohort_status").fill_null(0)
for col in ("eligible", "excluded"):
    if col in wide.columns:
        total = wide[col].sum()
        wide = wide.with_columns(
            (pl.col(col) / total * 100).round(1).alias(f"pct_{col}") if total else
            pl.lit(0.0).alias(f"pct_{col}")
        )
print(wide.sort("ADT_INTENT"))

## Validation against held-out signals

The label is medication-only by design, so it transfers to sites without
curated staging. That portability is worth nothing if the error rate is
unknown. The next few cells build two signals the classifier **never sees** and
measure agreement.

Neither is fed back into the label.

### Survival -- the primary go/no-go

`LOCALIZED_ADJUVANT` should show a markedly lower death rate than `METASTATIC`
at comparable follow-up. **If it does not, stop here** and do not use the label
downstream.

In [ ]:
survival = report_survival(labelled)
if survival.height == 0:
    print("[skip] no DEATH column joined -- cannot evaluate the survival contrast")
print(survival)

met = survival.filter(pl.col("ADT_INTENT") == "METASTATIC")
loc = survival.filter(pl.col("ADT_INTENT") == "LOCALIZED_ADJUVANT")
if met.height and loc.height:
    if loc["pct_died"][0] >= met["pct_died"][0]:
        print(
            "\nWARNING: LOCALIZED_ADJUVANT does not show better survival than "
            "METASTATIC. The label is not separating the populations -- "
            "do not use it downstream until this is resolved."
        )
    else:
        print(
            f"\nOK: adjuvant {loc['pct_died'][0]}% died vs "
            f"metastatic {met['pct_died'][0]}%."
        )

### Coded metastasis (ICD C77-C79, C7B)

Presence of a secondary-malignancy code in a patient labelled
`LOCALIZED_ADJUVANT` is a hard contradiction. Its *absence* is weak evidence --
metastasis coding is inconsistent -- so read the contradiction count, not the
agreement rate.

Codes are matched by prefix rather than the numeric-range regex used elsewhere
in the pipeline: that regex captures `^[A-Z](\d{2,3})`, giving 781 for a
dotless `C7810`, which silently fails its `77 <= n <= 79` test.

In [ ]:
if ICD_PATH.exists():
    icds = pl.read_csv(ICD_PATH, infer_schema_length=0)
    met_icd = compute_first_metastasis_icd_date(icds)
    labelled = labelled.join(met_icd, on="DFCI_MRN", how="left")

    print(report_against_metastasis_icd(labelled))

    contradictions = report_contradictions(labelled)
    print(f"\ncontradictions: {contradictions.height} adjuvant-labelled patients "
          f"carry a metastasis code")
    print(contradictions.head(20))
else:
    print(f"[skip] {ICD_PATH} not found -- run Stage 1 of 01_preprocessing.ipynb")

### PSA trajectory

A deep nadir (< 0.1) with no subsequent rise is the adjuvant pattern; a nadir
followed by a rise is biochemical progression. Only post-anchor PSA counts, and
the rise is measured strictly after the nadir so a high pre-nadir value is not
mistaken for progression.

Below-detection PSA is genuinely imputed to `0.0` upstream in `dfci_labs`, so
zeros here are real measurements rather than missing data.

Set `LONGITUDINAL_PATH` to the prediction data written by Stage 2/3. Left as
`None` this section is skipped.

In [ ]:
LONGITUDINAL_PATH = None  # e.g. DATA_ROOT / "longitudinal_prediction_data_adt.csv"

if LONGITUDINAL_PATH is not None and Path(LONGITUDINAL_PATH).exists():
    labs = pl.read_csv(LONGITUDINAL_PATH, infer_schema_length=0).with_columns(
        pl.col("LAB_VALUE").cast(pl.Float64, strict=False),
        pl.col("t_lab").cast(pl.Float64, strict=False),
    )
    # LAB_NAME is not spelled identically in every release -- check before filtering.
    print(labs.filter(pl.col("LAB_NAME").str.contains("(?i)psa|prostate"))
              ["LAB_NAME"].value_counts().head(10))

    psa = compute_psa_nadir_features(labs)
    labelled = labelled.join(psa, on="DFCI_MRN", how="left")
    print()
    print(_rate_report(labelled, ["PSA_DEEP_NADIR_NO_RISE"]))
else:
    print("[skip] set LONGITUDINAL_PATH to the Stage 2/3 prediction data to run this")

## Known failure modes

Two reports that exist to expose the label's weak points rather than assume
them away.

### ARPI-era drift

ARPIs moved into non-metastatic and high-risk localized disease over the study
period, so rule 2 gets less specific in later years. Watch for a `METASTATIC`
share that climbs with calendar year for reasons of practice pattern rather
than biology.

In [ ]:
by_year = report_by_adt_start_year(labelled)
print(
    by_year.pivot(values="n_patients", index="adt_start_year", on="ADT_INTENT")
    .fill_null(0)
    .sort("adt_start_year")
)

### Gap-threshold sensitivity

Episodes are reconstructed from refill spacing because `MEDICATIONS` has no end
date, dose, or days-supply -- only `MED_START_DT`. The 270-day default exceeds
the longest depot formulation plus refill slack, but no column exists to
validate it against, so its influence is measured rather than assumed
negligible.

Stable class counts across 180/270/365 mean the label does not hinge on the
threshold. Large swings mean intermittent therapy is common in this cohort and
the threshold needs clinical review.

In [ ]:
sensitivity = report_gap_sensitivity(meds, follow_up)
print(
    sensitivity.pivot(values="n_patients", index="ADT_INTENT", on="gap_threshold_days")
    .fill_null(0)
)

## Write the label

One row per ADT-exposed ICD-C61 patient with the label, the rule that produced
it, and every underlying feature, so any downstream use can audit or
re-threshold without recomputing. `ELIGIBLE` rides along so the file can be
narrowed to the modelled cohort with a filter rather than a re-run.

In [ ]:
out_path = OUT_DIR / "adt_intent_labels.csv"
labelled.write_csv(out_path)
print(f"wrote {labelled.height:,} labels to {out_path}")
print(f"  of which ELIGIBLE: {labelled.filter(pl.col('ELIGIBLE') == 1).height:,}")
print()
print(summarize_intent(labelled))

## Longitudinal labs and survival by intent

Everything above assigns and audits the label. This section asks what the
three groups actually look like over time -- testosterone and PSA from one year
before the ADT anchor to five years after, then Kaplan-Meier curves for death
and the three modelled endpoints.

These are held-out views. The classifier reads only `MEDICATIONS`, so no lab
value and no endpoint appears anywhere in its inputs.

**Requires Stage 2** (`cp.preprocess_labs`) for the `adt` arm, which writes the
longitudinal frame this section reads. Its `t_lab` is already days from
`TREATMENT_ANCHOR_DATE` -- negative before ADT, positive after -- so it is used
directly as the time axis with no re-derivation.

One caveat on population. The Stage 2 output is restricted to the eligible
survival cohort, so the trajectories and the three endpoint KMs cover eligible
patients only. The death KM is built separately from patient-status follow-up
and covers **every** ADT-exposed patient. Comparing the two death curves is
worth a moment: if they differ, the cohort exclusions are themselves selecting
on survival.

In [ ]:
import matplotlib.pyplot as plt

# Stage 2 output for the adt arm. Set to the path cp.make_endpoint_runs()
# assigned as the arm's input_csv.
LONGITUDINAL_ADT_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"

HAVE_LONGITUDINAL = Path(LONGITUDINAL_ADT_PATH).exists()
if HAVE_LONGITUDINAL:
    # load_longitudinal cross-checks TREATMENT_ANCHOR_DATE against each
    # patient's ADT_FIRST_DATE and refuses the arpi-arm file, whose anchor is
    # first ARPI/taxane exposure -- same column names, silently wrong origin.
    longitudinal = load_longitudinal(LONGITUDINAL_ADT_PATH, labels=labelled)
    print(f"longitudinal rows: {longitudinal.height:,}")
    print(f"patients with labs: {longitudinal['DFCI_MRN'].n_unique():,}")
else:
    longitudinal = None
    print(f"[skip] {LONGITUDINAL_ADT_PATH} not found -- run Stage 2 (cp.preprocess_labs) "
          f"for the adt arm to enable the trajectory and endpoint-KM cells")

### Testosterone and PSA trajectories

Each line is the median across patients in a 30-day bin, with the band showing
the IQR. A patient contributing several draws to one bin is averaged within
that bin first, so heavily-monitored metastatic patients do not weight a bin by
how often they were drawn.

**Testosterone tests the exposure.** Both groups genuinely receive ADT, so both
should castrate (below 50 ng/dL, dashed lines) within roughly three months.
They separate on the *right* of the plot instead: adjuvant testosterone should
recover toward eugonadal once a protocol-limited course ends, typically 6-24
months out, while metastatic stays suppressed indefinitely. A metastatic group
that recovers, or an adjuvant group that never does, means the duration signal
is not measuring what it claims to.

**PSA tests the disease.** Adjuvant is a deep nadir that stays down; metastatic
is a nadir followed by a rise as castration resistance emerges. PSA is on a log
axis because it spans orders of magnitude -- on a linear axis the adjuvant
group is squashed flat against zero by a few very high metastatic values.

Read the pre-anchor year too. Untreated metastatic disease usually presents
with a higher PSA than localized disease, so separation that is already visible
*before* ADT starts is real prognostic signal. If the groups are identical
before the anchor and only diverge after it, the label may be tracking
treatment duration alone.

Check coverage before reading either panel: testosterone is ordered far less
consistently than PSA, and a flat curve on thin data is missing data, not
biology.

In [ ]:
if HAVE_LONGITUDINAL:
    specs = [
        (TESTOSTERONE_LAB_NAME, "Testosterone (ng/dL)", False,
         (CASTRATE_NG_DL, CASTRATE_STRICT_NG_DL)),
        (PSA_LAB_NAME, "PSA (ng/mL, log scale)", True, ()),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    for ax, (lab, ylabel, log_scale, hlines) in zip(axes, specs):
        print(f"[{lab}] coverage:")
        print(summarize_trajectory_coverage(longitudinal, labelled, lab))
        print()

        traj = build_lab_trajectory(longitudinal, labelled, lab)
        if log_scale and traj.height:
            # Zeros are real below-detection values, but log10(0) is not
            # plottable -- floor for the axis only, after the median.
            traj = traj.with_columns(
                *[pl.col(c).clip(lower_bound=PSA_LOG_FLOOR) for c in ("median", "q1", "q3")]
            )
        plot_lab_trajectory(ax, traj, title=f"{lab} around ADT start",
                            ylabel=ylabel, log_scale=log_scale, hlines=hlines)
    fig.tight_layout()
    plt.show()
else:
    print("[skip] Stage 2 longitudinal data required")

### Kaplan-Meier: death, platinum, NEPC, AVPC

Death first, on the full ADT-exposed population. This is the primary go/no-go
restated as a curve: if `LOCALIZED_ADJUVANT` does not separate upward from
`METASTATIC`, the label is not doing its job.

The log-rank p-values are descriptive. These groups come from a derived label,
not randomization, so a small p-value says the label separates the outcome --
not that treatment intent caused the difference.

In [ ]:
death_km = build_death_km_input(labelled, follow_up)
print(summarize_km(death_km))
print()
print(logrank_by_intent(death_km))

fig, ax = plt.subplots(figsize=(7.5, 5.5))
overlay_km(
    ax, km_series_by_intent(death_km), colors=INTENT_COLORS,
    title="Overall survival by ADT intent (all ADT-exposed)",
    xlabel="Days from first ADT", ylabel="Survival probability",
)
ax.grid(alpha=0.2)
plt.show()

Now the three modelled endpoints, on the eligible cohort where they are
defined. Each is time from the ADT anchor to first platinum exposure, to the
adjudicated NEPC criterion, and to meeting AVPC respectively.

`METASTATIC` should reach all three sooner. These endpoints are what the
COMPASS models predict, so a label that fails to separate them is a label with
no bearing on the modelling problem -- whatever it does for survival.

Watch the event counts printed above each curve. NEPC and AVPC are rare, so an
adjuvant curve sitting flat at 1.0 may reflect a handful of events rather than
a real contrast.

In [ ]:
if HAVE_LONGITUDINAL:
    endpoints = [e for e in KM_ENDPOINTS if e != "death"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, endpoint in zip(axes, endpoints):
        km_input = build_km_input(longitudinal, labelled, endpoint)
        title = KM_ENDPOINTS[endpoint][2]
        print(f"[{endpoint}]")
        if km_input.height == 0:
            print(f"  no {endpoint} columns in the longitudinal file -- "
                  f"rebuild Stage 2 from a cohort carrying this endpoint\n")
            ax.set_title(f"{title} -- no data")
            ax.axis("off")
            continue
        print(summarize_km(km_input))
        print()
        overlay_km(
            ax, km_series_by_intent(km_input), colors=INTENT_COLORS,
            title=title, xlabel="Days from first ADT",
            ylabel="Event-free probability",
        )
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()
else:
    print("[skip] Stage 2 longitudinal data required")

## Using the label downstream

Nothing reads this automatically. To restrict a survival run to metastatic
patients, join on `DFCI_MRN` and filter before building prediction inputs:

```python
intent = pl.read_csv(OUT_DIR / "adt_intent_labels.csv")
metastatic = intent.filter(
    (pl.col("ADT_INTENT") == "METASTATIC") & (pl.col("ELIGIBLE") == 1)
)["DFCI_MRN"]
```

The `ELIGIBLE` filter matters: this file covers every ADT-exposed ICD-C61
patient, which is wider than any cohort the pipeline models. Joining it to a
survival run without that filter reintroduces patients the cohort excluded.

Two further cautions.

`INDETERMINATE` is a real class, not a bin for leftovers -- it is mostly
patients whose follow-up is too short to tell, and dropping them silently
removes recently-treated patients from the cohort. Decide deliberately whether
a given analysis keeps them.

Filtering the cohort by an ADT-history-derived label also narrows the
population every downstream estimate applies to, and interacts with the
existing exclusions (PARPi, pre-diagnosis platinum, competing cancers). Treat
it as a sensitivity analysis alongside the full cohort rather than a
replacement for it.